In [11]:
import pandas as pd
from pathlib import Path
import numpy as np
import json
from sklearn.preprocessing import OneHotEncoder,PowerTransformer
ROOT = Path.cwd().parent
motor='DATA_MODEL_Dev/Motor_data_models.csv'
cognitive='DATA_MODEL_Dev/Cognitive_data_models.csv'
sleep='DATA_MODEL_Dev/Sleep_data_models.csv'
data_split='DATA_RAW_MODELS/Data_split.json'

# DATA SPLIT

In [12]:
with open(ROOT/data_split, "r", encoding="utf-8") as archivo:
    data_subjects = json.load(archivo)

subjects_train=data_subjects['Train_80']
subjects_test=data_subjects['Test_20']

## Normal Data

### X_train and y_train

In [13]:
X_train_motor=pd.read_csv(ROOT/motor)
X_train_motor=X_train_motor[X_train_motor['subject_visit'].isin(subjects_train)]

X_train_cognitive=pd.read_csv(ROOT/cognitive)
X_train_cognitive=X_train_cognitive[X_train_cognitive['subject_visit'].isin(subjects_train)]
X_train_cognitive.drop(columns=['UPDRS_III_ProgressionType','MDS-UPDRS Total Score','MDS-UPDRS Part II Total Score',
                                'MDS-UPDRS Part I (Patient Questionnaire) Total Score','VISUOSPATIAL_EXECUTIVE','NAMING','ATTENTION',
                                'LANGUAGE','ABSTRACTION','DELAYED_RECALL','ORIENTATION'], inplace=True)

X_train_sleep=pd.read_csv(ROOT/sleep)
X_train_sleep=X_train_sleep[X_train_sleep['subject_visit'].isin(subjects_train)]
X_train_sleep.drop(columns=['UPDRS_III_ProgressionType','MDS-UPDRS Part II Total Score','MDS-UPDRS Part IV Total Score',
                            'MDS-UPDRS Part III Total Score','MDS-UPDRS Part I (Patient Questionnaire) Total Score',
                            'Sitting and reading','Watching TV','Sitting, inactive in a public place','As a passenger in a car for an hour',
                            'Lying down to rest in the afternoon','Sitting and talking to someone','Sitting quietly after lunch',
                            'In a car, while stopped in traffic','Vivid Dreams','Aggressive or Action-packed dreams','nocturnal behaviour',
                            'move arms/legs during sleep','hurt bed partner','speaking in sleep','sudden limb movements','complex movements',
                            'things fell down','my movements awake me','remember dreams','sleep is disturbed','Stroke','Head trauma',
                            'Parkinsonism','Restless Leg Syndrome (RLS)', 'Narcolepsy','Depression','Epilepsy','Inflammatory disease of the brain'], inplace=True)

X_train=pd.merge(X_train_motor, X_train_cognitive, on='subject_visit', how='inner')
X_train.drop(columns=['subject_visit'], inplace=True)

y_train=X_train_motor['UPDRS_III_ProgressionType']
X_train.drop(columns=['UPDRS_III_ProgressionType'], inplace=True)

print(X_train.shape,y_train.shape)


(728, 33) (728,)


### X_test and y_test

In [14]:
X_test_motor=pd.read_csv(ROOT/motor)
X_test_motor=X_test_motor[X_test_motor['subject_visit'].isin(subjects_test)]

X_test_cognitive=pd.read_csv(ROOT/cognitive)
X_test_cognitive=X_test_cognitive[X_test_cognitive['subject_visit'].isin(subjects_test)]
X_test_cognitive.drop(columns=['UPDRS_III_ProgressionType','MDS-UPDRS Total Score','MDS-UPDRS Part II Total Score',
                                'MDS-UPDRS Part I (Patient Questionnaire) Total Score','VISUOSPATIAL_EXECUTIVE','NAMING','ATTENTION',
                                'LANGUAGE','ABSTRACTION','DELAYED_RECALL','ORIENTATION'], inplace=True)

X_test_sleep=pd.read_csv(ROOT/sleep)
X_test_sleep=X_test_sleep[X_test_sleep['subject_visit'].isin(subjects_test)]
X_test_sleep.drop(columns=['UPDRS_III_ProgressionType','MDS-UPDRS Part II Total Score','MDS-UPDRS Part IV Total Score',
                            'MDS-UPDRS Part III Total Score','MDS-UPDRS Part I (Patient Questionnaire) Total Score',
                            'Sitting and reading','Watching TV','Sitting, inactive in a public place','As a passenger in a car for an hour',
                            'Lying down to rest in the afternoon','Sitting and talking to someone','Sitting quietly after lunch',
                            'In a car, while stopped in traffic','Vivid Dreams','Aggressive or Action-packed dreams','nocturnal behaviour',
                            'move arms/legs during sleep','hurt bed partner','speaking in sleep','sudden limb movements','complex movements',
                            'things fell down','my movements awake me','remember dreams','sleep is disturbed','Stroke','Head trauma',
                            'Parkinsonism','Restless Leg Syndrome (RLS)', 'Narcolepsy','Depression','Epilepsy','Inflammatory disease of the brain'], inplace=True)

X_test=pd.merge(X_test_motor, X_test_cognitive, on='subject_visit', how='inner')
X_test.drop(columns=['subject_visit'], inplace=True)

y_test=X_test_motor['UPDRS_III_ProgressionType']
X_test.drop(columns=['UPDRS_III_ProgressionType'], inplace=True)

print(X_test.shape,y_test.shape)

(183, 33) (183,)


## Data Agrupations Stability-Improvement VS Worsening

In [15]:
y_train_2_IS_W= y_train.copy()
y_train_2_IS_W=y_train_2_IS_W.replace({0:0,-1:1,1:1})

y_test_2_IS_W= y_test.copy()
y_test_2_IS_W=y_test_2_IS_W.replace({0:0,-1:1,1:1})

## Data Agrupations Stability VS Improvement-Worsening

In [16]:
y_train_2_S_IW= y_train.copy()
y_train_2_S_IW=y_train_2_S_IW.replace({0:0,-1:0,1:1})

y_test_2_S_IW= y_test.copy()
y_test_2_S_IW=y_test_2_S_IW.replace({0:0,-1:0,1:1})

## Feature Engineering

In [17]:
X_train_fe=X_train.copy()

X_train_fe['motor_burden']=X_train_fe['MDS-UPDRS Part III Total Score']+X_train_fe['MDS-UPDRS Part IV Total Score']
X_train_fe['QoL_burden']=X_train_fe['MDS-UPDRS Part I (Patient Questionnaire) Total Score']+X_train_fe['MDS-UPDRS Part II Total Score']
X_train_fe['motor_ratio']=X_train_fe['motor_burden']/(X_train_fe['QoL_burden']+1)


X_train_fe['motor_cog_ratio_qol']=X_train_fe['MDS-UPDRS Part I (Patient Questionnaire) Total Score']/(X_train_fe['MDS-UPDRS Part II Total Score']+1)

X_train_fe['UPDRS1_weight']=X_train_fe['MDS-UPDRS Part I (Patient Questionnaire) Total Score']/(X_train_fe['MDS-UPDRS Total Score']+1)
X_train_fe['UPDRS2_weight']=X_train_fe['MDS-UPDRS Part II Total Score']/(X_train_fe['MDS-UPDRS Total Score']+1)
X_train_fe['UPDRS3_weight']=X_train_fe['MDS-UPDRS Part III Total Score']/(X_train_fe['MDS-UPDRS Total Score']+1)

X_train_fe['Class3_H&Y']=X_train_fe['UPDRS_III_Class']/(X_train_fe['3.21 HOEHN AND YAHR STAGE']+0.01)
X_train_fe['Functional_impairment']= X_train_fe['SCHWAB & ENGLAND ADL'].apply(lambda x: 100 - x)


encoder = OneHotEncoder(sparse_output=False)

resultados = encoder.fit_transform(X_train_fe[['3.21 HOEHN AND YAHR STAGE']])
columnas_encoder = encoder.get_feature_names_out(['3.21 HOEHN AND YAHR STAGE'])

df_resultados = pd.DataFrame(
    resultados,
    columns=columnas_encoder,
    index=X_train_fe.index
)

X_train_fe = pd.concat([X_train_fe, df_resultados], axis=1)
X_train_fe.drop(columns=['3.21 HOEHN AND YAHR STAGE'], inplace=True)

X_train_fe.head()


,SCHWAB & ENGLAND ADL,MDS-UPDRS Part I (Patient Questionnaire) Total Score,MDS-UPDRS Part II Total Score,Does participant have DBS,MDS-UPDRS Part III Total Score,MDS-UPDRS Part IV Total Score,DBS_Transition_Visit,DBS_Post_Transition,MDS-UPDRS Total Score,UPDRS_I_Class,...,UPDRS1_weight,UPDRS2_weight,UPDRS3_weight,Class3_H&Y,Functional_impairment,3.21 HOEHN AND YAHR STAGE_0,3.21 HOEHN AND YAHR STAGE_1,3.21 HOEHN AND YAHR STAGE_2,3.21 HOEHN AND YAHR STAGE_3,3.21 HOEHN AND YAHR STAGE_4
0,90,6,5.0,0,35,0.0,0,0.0,46.0,0,...,0.127660,0.106383,0.744681,0.497512,10,0.0,0.0,1.0,0.0,0.0
1,85,9,10.0,0,53,0.0,0,0.0,72.0,0,...,0.123288,0.136986,0.726027,0.332226,15,0.0,0.0,0.0,1.0,0.0
2,80,11,16.0,0,18,7.0,0,0.0,52.0,1,...,0.207547,0.301887,0.339623,0.000000,20,0.0,0.0,1.0,0.0,0.0
3,90,8,8.0,0,34,0.0,0,0.0,50.0,0,...,0.156863,0.156863,0.666667,0.497512,10,0.0,0.0,1.0,0.0,0.0
4,85,4,9.0,0,16,0.0,0,0.0,29.0,0,...,0.133333,0.300000,0.533333,0.000000,15,0.0,0.0,1.0,0.0,0.0


In [18]:
X_test_fe=X_test.copy()



X_test_fe['motor_burden']=X_test_fe['MDS-UPDRS Part III Total Score']+X_test_fe['MDS-UPDRS Part IV Total Score']
X_test_fe['QoL_burden']=X_test_fe['MDS-UPDRS Part I (Patient Questionnaire) Total Score']+X_test_fe['MDS-UPDRS Part II Total Score']
X_test_fe['motor_ratio']=X_test_fe['motor_burden']/(X_test_fe['QoL_burden']+1)


X_test_fe['motor_cog_ratio_qol']=X_test_fe['MDS-UPDRS Part I (Patient Questionnaire) Total Score']/(X_test_fe['MDS-UPDRS Part II Total Score']+1)

X_test_fe['UPDRS1_weight']=X_test_fe['MDS-UPDRS Part I (Patient Questionnaire) Total Score']/(X_test_fe['MDS-UPDRS Total Score']+1)
X_test_fe['UPDRS2_weight']=X_test_fe['MDS-UPDRS Part II Total Score']/(X_test_fe['MDS-UPDRS Total Score']+1)
X_test_fe['UPDRS3_weight']=X_test_fe['MDS-UPDRS Part III Total Score']/(X_test_fe['MDS-UPDRS Total Score']+1)

X_test_fe['Class3_H&Y']=X_test_fe['UPDRS_III_Class']/(X_test_fe['3.21 HOEHN AND YAHR STAGE']+0.01)
X_test_fe['Functional_impairment']= X_test_fe['SCHWAB & ENGLAND ADL'].apply(lambda x: 100 - x)


encoder = OneHotEncoder(sparse_output=False)

resultados = encoder.fit_transform(X_test_fe[['3.21 HOEHN AND YAHR STAGE']])
columnas_encoder = encoder.get_feature_names_out(['3.21 HOEHN AND YAHR STAGE'])

X_test_fe['3.21 HOEHN AND YAHR STAGE_0']=0
df_resultados = pd.DataFrame(
    resultados,
    columns=columnas_encoder,
    index=X_test_fe.index
)

X_test_fe = pd.concat([X_test_fe, df_resultados], axis=1)
X_test_fe['3.21 HOEHN AND YAHR STAGE_4']=0
X_test_fe.drop(columns=['3.21 HOEHN AND YAHR STAGE'], inplace=True)


X_test_fe.head()


,SCHWAB & ENGLAND ADL,MDS-UPDRS Part I (Patient Questionnaire) Total Score,MDS-UPDRS Part II Total Score,Does participant have DBS,MDS-UPDRS Part III Total Score,MDS-UPDRS Part IV Total Score,DBS_Transition_Visit,DBS_Post_Transition,MDS-UPDRS Total Score,UPDRS_I_Class,...,UPDRS1_weight,UPDRS2_weight,UPDRS3_weight,Class3_H&Y,Functional_impairment,3.21 HOEHN AND YAHR STAGE_0,3.21 HOEHN AND YAHR STAGE_1,3.21 HOEHN AND YAHR STAGE_2,3.21 HOEHN AND YAHR STAGE_3,3.21 HOEHN AND YAHR STAGE_4
0,90,5,5.0,0,18,5.0,0,0.0,33.0,0,...,0.147059,0.147059,0.529412,0.0,10,0,0.0,1.0,0.0,0
1,90,4,1.0,0,15,4.0,0,0.0,24.0,0,...,0.160000,0.040000,0.600000,0.0,10,0,0.0,1.0,0.0,0
2,80,2,7.0,0,13,1.0,0,0.0,23.0,0,...,0.083333,0.291667,0.541667,0.0,20,0,0.0,1.0,0.0,0
3,90,2,3.0,0,15,0.0,0,0.0,20.0,0,...,0.095238,0.142857,0.714286,0.0,10,0,0.0,1.0,0.0,0
4,80,2,4.0,0,9,0.0,0,0.0,15.0,0,...,0.125000,0.250000,0.562500,0.0,20,0,0.0,1.0,0.0,0


In [19]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn import svm
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, BernoulliNB, GaussianNB
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.feature_selection import SequentialFeatureSelector
from imblearn.pipeline import Pipeline 
from imblearn.over_sampling import RandomOverSampler, SMOTE
from sklearn.base import clone

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import make_scorer, recall_score, confusion_matrix,f1_score,precision_score,balanced_accuracy_score


def specificity_weighted(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    n_classes = cm.shape[0]

    total_samples = np.sum(cm)
    specificities = []
    supports = []

    for k in range(n_classes):
        TP = cm[k, k]
        FN = np.sum(cm[k, :]) - TP
        FP = np.sum(cm[:, k]) - TP
        TN = total_samples - (TP + FN + FP)

        spec_k = TN / (TN + FP) if (TN + FP) > 0 else 0
        specificities.append(spec_k)
        supports.append(np.sum(cm[k, :]))

    return np.average(specificities, weights=supports)


def g_mean(y_true, y_pred):
    sens = recall_score(y_true, y_pred, average="macro")
    spec = specificity_weighted(y_true, y_pred)
    return np.sqrt(sens * spec)




def cv_results_to_row(results, modelo_name, parameters, sep=" ± "):

    row = {
        "Model": modelo_name,
        "Parameters": parameters
    }

    for k in results:
        if k.startswith("train_") or k.startswith("test_"):
            prefix, metric = k.split("_", 1)

            mean = results[k].mean()
            

            col_name = f"{prefix}_{metric}"
            row[col_name] = f"{mean:.4f}"

    return pd.DataFrame([row])


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "acc": "accuracy",
    "bal_acc": make_scorer(balanced_accuracy_score),
    "f1_macro": make_scorer(f1_score, average="macro"),
    "f1_weighted": make_scorer(f1_score, average="weighted"),
    "prec_macro": make_scorer(precision_score, average="macro", zero_division=0),
    "rec_macro": make_scorer(recall_score, average="macro", zero_division=0),
    "specificity_weighted": make_scorer(specificity_weighted),
    "g_mean": make_scorer(g_mean),
    
}


In [20]:
# 1) Dummy model (baseline)
Dummy_model = DummyClassifier(
    strategy="most_frequent",
    random_state=42
)
# 2) Pipeline
pipe_dummy = Pipeline([
    ("clf", Dummy_model)
])

# 3) Cross-validation
results_dummy = cross_validate(
    pipe_dummy,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

# 4) Convertir resultados a fila
df_dummy1 = cv_results_to_row(
    results_dummy,
    modelo_name="DummyClassifier",
    parameters=" 3 Classes/ Most Frequent",
    sep=" ± "
)

# 5) Para 2 Classes IS vs W
results_dummy_2_IS_W = cross_validate(
    pipe_dummy,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_dummy2 = cv_results_to_row(
    results_dummy_2_IS_W,
    modelo_name="DummyClassifier",
    parameters="IS vs W / 2 Classes/ Most Frequent / Unbalanced",
    sep=" ± "
)

# 6) Para 2 Classes S vs IW
results_dummy_2_S_IW = cross_validate(
    pipe_dummy,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_dummy3 = cv_results_to_row(
    results_dummy_2_S_IW,
    modelo_name="DummyClassifier",
    parameters="S vs IW / 2 Classes/ Most Frequent / Unbalanced",
    sep=" ± "
)
# 7) Balanced IS vs W with SMOTE
pipe_dummy_balanced = Pipeline([
    ('SMOTE', SMOTE(random_state=42)),
    ("clf", DummyClassifier())])

results_dummy_2_IS_W_balanced = cross_validate(
    pipe_dummy_balanced,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df_dummy4 = cv_results_to_row(
    results_dummy_2_IS_W_balanced,
    modelo_name="DummyClassifier",
    parameters="IS vs W / 2 Classes/ Most Frequent / SMOTE",
    sep=" ± "
)

# 8) Balanced S vs IW SMOTE
results_dummy_2_S_IW_balanced = cross_validate(
    pipe_dummy_balanced,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_dummy5 = cv_results_to_row(
    results_dummy_2_S_IW_balanced,
    modelo_name="DummyClassifier",
    parameters="S vs IW / 2 Classes/ Most Frequent / SMOTE",
    sep=" ± "
)

# 9) Balanced IS VS W with RandomOverSampler
pipe_dummy_ros = Pipeline([
    ('ROS', RandomOverSampler(random_state=42)),
    ("clf", DummyClassifier())])

results_dummy_2_IS_W_balanced_ros = cross_validate(
    pipe_dummy_ros,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_dummy6 = cv_results_to_row(
    results_dummy_2_IS_W_balanced_ros,
    modelo_name="DummyClassifier",
    parameters="IS vs W / 2 Classes/ Most Frequent / RandomOverSampler",
    sep=" ± "
)

#10) Balanced S vs IW with RandomOverSampler
results_dummy_2_S_IW_balanced_ros = cross_validate(
    pipe_dummy_ros,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_dummy7 = cv_results_to_row(
    results_dummy_2_S_IW_balanced_ros,
    modelo_name="DummyClassifier",
    parameters="S vs IW / 2 Classes/ Most Frequent / RandomOverSampler",
    sep=" ± "
)



# Combinar todos los resultados en un solo DataFrame
df_all_dummies = pd.concat([df_dummy1, df_dummy2, df_dummy3,df_dummy4,df_dummy5,df_dummy6,df_dummy7], ignore_index=True)
df_all_dummies



,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,DummyClassifier,3 Classes/ Most Frequent,0.3956,0.3956,0.3333,0.3333,0.1890,0.1890,0.2243,0.2243,0.1319,0.1319,0.3333,0.3333,0.6044,0.6044,0.4489,0.4488
1,DummyClassifier,IS vs W / 2 Classes/ Most Frequent / Unbalanced,0.6044,0.6044,0.5000,0.5000,0.3767,0.3767,0.4554,0.4554,0.3022,0.3022,0.5000,0.5000,0.3956,0.3956,0.4447,0.4447
2,DummyClassifier,S vs IW / 2 Classes/ Most Frequent / Unbalanced,0.7527,0.7527,0.5000,0.5000,0.4295,0.4295,0.6466,0.6466,0.3764,0.3764,0.5000,0.5000,0.2473,0.2473,0.3516,0.3516
3,DummyClassifier,IS vs W / 2 Classes/ Most Frequent / SMOTE,0.3956,0.3956,0.5000,0.5000,0.2835,0.2835,0.2243,0.2243,0.1978,0.1978,0.5000,0.5000,0.6044,0.6044,0.5497,0.5497
4,DummyClassifier,S vs IW / 2 Classes/ Most Frequent / SMOTE,0.7527,0.7527,0.5000,0.5000,0.4295,0.4295,0.6466,0.6466,0.3764,0.3764,0.5000,0.5000,0.2473,0.2473,0.3516,0.3516
5,DummyClassifier,IS vs W / 2 Classes/ Most Frequent / RandomOve...,0.3956,0.3956,0.5000,0.5000,0.2835,0.2835,0.2243,0.2243,0.1978,0.1978,0.5000,0.5000,0.6044,0.6044,0.5497,0.5497
6,DummyClassifier,S vs IW / 2 Classes/ Most Frequent / RandomOve...,0.7527,0.7527,0.5000,0.5000,0.4295,0.4295,0.6466,0.6466,0.3764,0.3764,0.5000,0.5000,0.2473,0.2473,0.3516,0.3516
